<a href="https://colab.research.google.com/github/ryanatberkeley/aeesp-grand-challenges-nlp-pipeline/blob/main/aseep_cleanedcorpus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OpenAlex Bibliographic Corpus Builder

This notebook queries the OpenAlex API to collect publications for a researcher and build a structured corpus containing:

- titles
- abstracts
- keywords
- authors
- publication metadata

In [ ]:
try:
    import pyalex
except ImportError:
    !pip install pyalex
    import pyalex

## Import libraries

In [ ]:
import pandas as pd
from pyalex import Works, Authors, Institutions
import time
from google.colab import files
import io
import difflib

## Upload AEESP member CSV


In [ ]:
print("Please upload your 'aeesp_members_short.csv' file:")
uploaded = files.upload()

file_name = list(uploaded.keys())[0]
df_authors = pd.read_csv(io.BytesIO(uploaded[file_name]))

df_authors.columns = df_authors.columns.str.strip().str.replace('\n', '')

print("\n--- Diagnostic Check ---")
print("Columns found in your CSV:", df_authors.columns.tolist())
print("------------------------\n")

Please upload your 'aeesp_members_short.csv' file:


## Preview uploaded data

In [ ]:
df_authors.head()

## Initialize storage for author dataframes

In [ ]:
all_author_dfs = {}

## Helper functions

These functions reconstruct OpenAlex abstracts and filter alternate author names.

In [ ]:
def reconstruct_abstract(inverted_index):
    if not inverted_index:
        return "No abstract available"

    max_index = max(max(indices) for indices in inverted_index.values())
    abstract_list = [""] * (max_index + 1)

    for word, indices in inverted_index.items():
        for index in indices:
            abstract_list[index] = word

    return " ".join(abstract_list)


def string_similarity(s1, s2):
    return difflib.SequenceMatcher(None, str(s1).lower(), str(s2).lower()).ratio()


def clean_alternative_names(primary_name, alternatives, threshold=0.5):
    if not alternatives:
        return []

    cleaned = []
    for alt in alternatives:
        if string_similarity(primary_name, alt) >= threshold:
            cleaned.append(alt)

    return cleaned

## Retrieve works for each AEESP member

For each row in the CSV:

1. Search OpenAlex for the institution
2. Search OpenAlex for the author within that institution
3. Retrieve all works associated with the author
4. Extract publication metadata
5. Reconstruct abstracts
6. Keep filtered alternate author names
7. Remove duplicate works
8. Store results as a dataframe

In [ ]:
for index, row in df_authors.iterrows():
    first_name = str(row['First Name']).strip() if 'First Name' in df_authors.columns else ''
    last_name = str(row['Last Name']).strip() if 'Last Name' in df_authors.columns else ''

    # 1. Determine and Clean Institution
    if 'Institution' in df_authors.columns:
        raw_institution = str(row['Institution']).strip()
    elif 'Job Institution' in df_authors.columns:
        raw_institution = str(row['Job Institution']).strip()
    else:
        print("Error: Could not find an 'Institution' column.")
        break

    # Clean the institution string to handle pipes and typos
    institution = raw_institution.split('|')[0].strip()
    # TESTING (just seeing what the issue is)
    if institution == "University at Nebraska - Lincoln":
        institution = "University of Nebraska-Lincoln"

    print(f"Processing: {first_name} {last_name} at {institution}...")

    try:
        # Search for Institution
        inst_search = Institutions().search(institution).get()
        if not inst_search:
            print(f"  -> Institution '{institution}' not found. Skipping.")
            continue

        inst_id = inst_search[0]['id']

        # 2. Try exact name search first
        auth_search = Authors().search(f"{first_name} {last_name}").filter(
            affiliations={"institution": {"id": inst_id}}
        ).get()

        # 3. Fallback search (First name + Last name only)
        if not auth_search:
            short_first = first_name.split()[0]
            short_last = last_name.split()[-1]
            print(f"  -> Exact name failed. Retrying as '{short_first} {short_last}'...")
            auth_search = Authors().search(f"{short_first} {short_last}").filter(
                affiliations={"institution": {"id": inst_id}}
            ).get()

        if not auth_search:
            print(f"  -> Author '{first_name} {last_name}' not found after fallback. Skipping.")
            continue

        # 4. Author Disambiguation (Check research concepts)
        auth = None
        for potential_author in auth_search:
            concepts = [c['display_name'].lower() for c in potential_author.get('x_concepts', [])]

            # Check if their work aligns with Environmental Engineering
            concept_keywords = ['environmental', 'water', 'engineering', 'chemistry', 'wastewater', 'civil']
            if any(keyword in " ".join(concepts) for keyword in concept_keywords):
                auth = potential_author
                break

        if not auth:
            print(f"  -> Warning: Using default profile for {first_name} {last_name}. Verify concepts.")
            auth = auth_search[0]

        author_id = auth['id']
        matched_author_name = auth.get('display_name')
        matched_author_orcid = auth.get('orcid')

        display_name_alternatives = clean_alternative_names(
            matched_author_name,
            auth.get('display_name_alternatives', [])
        )

        # 5. Retrieve works
        works_iterator = Works().filter(
            authorships={"author": {"id": author_id}}
        ).paginate(per_page=100)

        works_data = []

        for page in works_iterator:
            for work in page:
                primary_loc = work.get('primary_location') or {}
                source = primary_loc.get('source') or {}
                publication_name = source.get('display_name')

                work_type = work.get('type')
                pub_lower = publication_name.lower() if publication_name else ""

                non_archival_keywords = [
                  'biorxiv',
                  'medrxiv',
                  'zenodo',
                  'conference',
                  'abstract',
                  'thesis',
                  'dissertation'
                ]

                if work_type in ['preprint', 'dissertation']:
                    continue

                if any(keyword in pub_lower for keyword in non_archival_keywords):
                    continue

                biblio = work.get('biblio') or {}
                volume = biblio.get('volume')
                issue = biblio.get('issue')
                first_page = biblio.get('first_page')
                last_page = biblio.get('last_page')

                keywords_list = work.get('keywords') or []
                keywords = ", ".join([k.get('display_name', '') for k in keywords_list])

                # Abstract extraction
                abstract_inverted_index = work.get('abstract_inverted_index')
                has_abstract = abstract_inverted_index is not None
                abstract = reconstruct_abstract(abstract_inverted_index)

                works_data.append({
                    'id': work.get('id'),
                    'doi': work.get('doi'),
                    'title': work.get('title'),
                    'publication_year': work.get('publication_year'),
                    'publication_name': publication_name,
                    'volume': volume,
                    'issue': issue,
                    'first_page': first_page,
                    'last_page': last_page,
                    'keywords': keywords,
                    'open_access': work.get('open_access', {}).get('is_oa'),
                    'abstract': abstract,
                    'has_abstract': has_abstract,
                    'matched_author_name': matched_author_name,
                    'matched_author_orcid': matched_author_orcid,
                    'display_name_alternatives': "; ".join(display_name_alternatives)
                })

        df = pd.DataFrame(works_data)

        if not df.empty:
            df = df.sort_values(
              by=['has_abstract', 'publication_year'],
              ascending=[False, False]
            )

            df = df.drop_duplicates(subset=['id'], keep='first')

            if 'doi' in df.columns:
                df = df.drop_duplicates(subset=['doi'], keep='first')

            df = df.drop_duplicates(subset=['title'], keep='first')

        safe_last = last_name.lower().replace(' ', '').replace('-', '')
        safe_first = first_name.lower().replace(' ', '').replace('-', '')

        df_variable_name = f"works_df_{safe_last}_{safe_first}"

        globals()[df_variable_name] = df
        all_author_dfs[df_variable_name] = df

        print(f"  -> Success! Created `{df_variable_name}` with {len(df)} unique publications.")

    except Exception as e:
        print(f"  -> Error processing {first_name} {last_name}: {e}")

    time.sleep(1)

print("\nFinished processing all authors!")

## Export all author data

Each dataframe beginning with `works_df_` will be saved as a CSV file and packaged into a zip archive.

In [ ]:
import os
import zipfile
from google.colab import files

print("Gathering and zipping all author data...")

zip_filename = "all_authors_works.zip"

with zipfile.ZipFile(zip_filename, 'w') as zipf:

    for var_name in list(globals().keys()):

        if var_name.startswith('works_df_'):

            df = globals()[var_name]

            csv_filename = f"{var_name}.csv"

            df.to_csv(csv_filename, index=False)

            zipf.write(csv_filename)

            os.remove(csv_filename)

print(f"Successfully created {zip_filename}! Triggering download...")

files.download(zip_filename)